# GACS Video Generator

**Purpose**: Generate new short videos using GACS mood coordinates via scene splicing.

## Pipeline Overview
1. Load GACS dataset from `data/embeddings/gacs_dataset.csv`
2. Cluster mood vectors into K clusters
3. For each cluster centroid, generate GACS videos
4. Generate baseline (random) videos for comparison
5. Render videos and save metadata

## Prerequisites
- Run `gacs_dataset_builder.ipynb` first to generate embeddings
- Ensure `data/embeddings/gacs_dataset.csv` exists

## 1. Setup & Configuration

In [ ]:
# Install dependencies (run once)
# !pip install opencv-python moviepy numpy scipy pandas matplotlib scikit-learn anthropic tqdm

In [ ]:
import json
import random
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import anthropic
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from moviepy.editor import VideoFileClip, concatenate_videoclips
from moviepy.video.fx.all import fadein, fadeout
from sklearn.cluster import KMeans
from tqdm.notebook import tqdm

print("All dependencies loaded successfully!")

In [ ]:
# Configuration
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
EMBEDDINGS_DIR = DATA_DIR / "embeddings"
SCENES_DIR = DATA_DIR / "scenes"
GENERATED_DIR = DATA_DIR / "generated"
RAW_VIDEOS_DIR = DATA_DIR / "raw_videos"

# Create output directories
GACS_OUTPUT_DIR = GENERATED_DIR / "gacs"
BASELINE_OUTPUT_DIR = GENERATED_DIR / "baseline"
GACS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Clustering settings
DEFAULT_K_CLUSTERS = 5

# Video generation settings
TARGET_DURATION = 30  # seconds
MIN_SCENES = 5
MAX_SCENES = 8
OUTPUT_FPS = 30
TRANSITION_DURATION = 0.5

# Claude API
CLAUDE_MODEL = "claude-sonnet-4-20250514"

print(f"Base directory: {BASE_DIR}")
print(f"GACS output: {GACS_OUTPUT_DIR}")
print(f"Baseline output: {BASELINE_OUTPUT_DIR}")

## 2. Load GACS Dataset

In [ ]:
@dataclass
class SceneData:
    """Scene data from GACS dataset."""
    image_id: str
    video_id: str
    scene_id: str
    keyframe_path: str
    mood_vector: np.ndarray
    mood_words: List[str]
    style_words: List[str]

@dataclass
class GeneratedVideoMetadata:
    """Metadata for generated video."""
    video_id: str
    group: str  # "gacs" or "baseline"
    mood_vector: List[float]
    mood_words: List[str]
    prompt_used: str
    source_scenes: List[str]
    created_at: str
    duration: float

print("Data classes defined.")

In [ ]:
# Load GACS dataset
dataset_path = EMBEDDINGS_DIR / "gacs_dataset.csv"

if dataset_path.exists():
    gacs_df = pd.read_csv(dataset_path)
    print(f"Loaded {len(gacs_df)} entries from GACS dataset")

    # Parse mood vectors back to numpy arrays
    def parse_mood_vector(v):
        return np.array([float(x) for x in v.split(',')])

    gacs_df['mood_vector_array'] = gacs_df['mood_vector'].apply(parse_mood_vector)

    # Convert to SceneData objects
    scenes_data = []
    for _, row in gacs_df.iterrows():
        scene = SceneData(
            image_id=row['image_id'],
            video_id=row['video_id'],
            scene_id=row['scene_id'],
            keyframe_path=row['keyframe_path'],
            mood_vector=row['mood_vector_array'],
            mood_words=[w.strip() for w in row['mood_words'].split(',')],
            style_words=[w.strip() for w in row['style_words'].split(',')]
        )
        scenes_data.append(scene)

    # Create lookup by scene_id
    scene_lookup = {s.scene_id: s for s in scenes_data}

    print("\nSample scene:")
    sample = scenes_data[0]
    print(f"  Scene ID: {sample.scene_id}")
    print(f"  Mood words: {sample.mood_words}")
    print(f"  Style words: {sample.style_words}")
else:
    print(f"ERROR: Dataset not found at {dataset_path}")
    print("Please run gacs_dataset_builder.ipynb first.")
    scenes_data = []
    scene_lookup = {}

## 3. Mood Vector Clustering

In [ ]:
def cluster_mood_vectors(scenes: List[SceneData], k: int = DEFAULT_K_CLUSTERS) -> Tuple[KMeans, Dict]:
    """
    Cluster mood vectors into K clusters.
    
    Args:
        scenes: List of SceneData objects
        k: Number of clusters
    
    Returns:
        Tuple of (KMeans model, cluster info dict)
    """
    if not scenes:
        return None, {}

    # Stack mood vectors
    vectors = np.stack([s.mood_vector for s in scenes])

    # Fit KMeans
    print(f"Clustering {len(vectors)} mood vectors into {k} clusters...")
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(vectors)

    # Build cluster info
    cluster_info = {}
    for i in range(k):
        cluster_scenes = [s for s, l in zip(scenes, labels) if l == i]

        # Aggregate mood words for this cluster
        all_mood_words = []
        for s in cluster_scenes:
            all_mood_words.extend(s.mood_words)

        # Get top mood words
        word_counts = pd.Series(all_mood_words).value_counts()
        top_words = word_counts.head(5).index.tolist()

        cluster_info[i] = {
            'centroid': kmeans.cluster_centers_[i],
            'num_scenes': len(cluster_scenes),
            'scene_ids': [s.scene_id for s in cluster_scenes],
            'top_mood_words': top_words
        }

    return kmeans, cluster_info

# Cluster the dataset
if scenes_data:
    kmeans_model, cluster_info = cluster_mood_vectors(scenes_data, k=DEFAULT_K_CLUSTERS)

    print("\nCluster Summary:")
    print("-" * 60)
    for i, info in cluster_info.items():
        print(f"Cluster {i}: {info['num_scenes']} scenes")
        print(f"  Top mood words: {info['top_mood_words']}")
else:
    kmeans_model = None
    cluster_info = {}

In [ ]:
def visualize_clusters(scenes: List[SceneData], kmeans: KMeans):
    """
    Visualize mood clusters using t-SNE.
    """
    from sklearn.manifold import TSNE

    vectors = np.stack([s.mood_vector for s in scenes])
    labels = kmeans.labels_

    # Apply t-SNE
    print("Applying t-SNE...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors)-1))
    vectors_2d = tsne.fit_transform(vectors)

    # Also transform centroids
    centroids_2d = tsne.fit_transform(kmeans.cluster_centers_)

    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))

    colors = plt.cm.tab10(np.linspace(0, 1, kmeans.n_clusters))

    for i in range(kmeans.n_clusters):
        mask = labels == i
        ax.scatter(vectors_2d[mask, 0], vectors_2d[mask, 1],
                   c=[colors[i]], label=f'Cluster {i}', alpha=0.6, s=50)

    ax.set_xlabel('t-SNE Dimension 1')
    ax.set_ylabel('t-SNE Dimension 2')
    ax.set_title('GACS Mood Clusters')
    ax.legend()

    plt.tight_layout()
    plt.savefig(EMBEDDINGS_DIR / 'mood_clusters.png', dpi=150)
    plt.show()

if scenes_data and kmeans_model:
    visualize_clusters(scenes_data, kmeans_model)

## 4. GACS Generation Prompt

In [ ]:
# Initialize Claude client
client = anthropic.Anthropic()

GACS_GENERATION_PROMPT = """You are a GACS video generator.
Given:
- target mood words: {mood_words}
- scene pool with mood embeddings

Task:
Select 5 to 8 scenes whose mood vectors are closest to target.
Arrange into a 30-second emotional narrative.

Rules:
- Maintain emotional continuity.
- Avoid abrupt visual jumps.
- No duplicate scenes.
Output:
Ordered list of scene IDs.

Available scenes (scene_id: mood_words):
{scene_list}

Respond with ONLY a comma-separated list of scene IDs in the order they should appear.
Example: scene_001, scene_005, scene_003, scene_012, scene_008"""

BASELINE_PROMPT = """Generate a random but visually coherent sequence of scenes.
Ignore mood vectors.

Select 5 to 8 scenes from the pool below.
Prioritize visual variety and coherent transitions.

Available scenes (scene_id: style_words):
{scene_list}

Respond with ONLY a comma-separated list of scene IDs.
Example: scene_001, scene_005, scene_003, scene_012, scene_008"""

print("Generation prompts defined.")

In [ ]:
def enforce_diversity(scene_ids, max_per_video=2):
    """Enforce max scenes per source video."""
    video_counts = {}
    diverse_ids = []
    for sid in scene_ids:
        video_id = sid.rsplit('_s', 1)[0]
        video_counts[video_id] = video_counts.get(video_id, 0) + 1
        if video_counts[video_id] <= max_per_video:
            diverse_ids.append(sid)
    return diverse_ids


def select_scenes_gacs(target_mood_words: List[str],
                       scenes: List[SceneData],
                       num_candidates: int = 30) -> Tuple[List[str], str]:
    """
    Select scenes using GACS method (mood-based).
    
    Args:
        target_mood_words: Target mood words for the video
        scenes: Pool of available scenes
        num_candidates: Number of candidates to show Claude
    
    Returns:
        Tuple of (ordered scene IDs, prompt used)
    """
    # Pre-filter: find scenes closest to target mood
    # Use simple word overlap as a heuristic
    target_set = set(w.lower() for w in target_mood_words)

    scored_scenes = []
    for scene in scenes:
        scene_words = set(w.lower() for w in scene.mood_words)
        overlap = len(target_set & scene_words)
        scored_scenes.append((scene, overlap))

    # Sort by overlap and take top candidates
    scored_scenes.sort(key=lambda x: x[1], reverse=True)
    candidates = [s[0] for s in scored_scenes[:num_candidates]]

    # Build scene list for prompt
    scene_list = "\n".join([
        f"{s.scene_id}: {', '.join(s.mood_words)}"
        for s in candidates
    ])

    # Format prompt
    prompt = GACS_GENERATION_PROMPT.format(
        mood_words=', '.join(target_mood_words),
        scene_list=scene_list
    )

    # Call Claude
    try:
        message = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}]
        )
        response = message.content[0].text.strip()

        # Parse scene IDs
        scene_ids = [s.strip() for s in response.split(',')]

        # Validate scene IDs
        valid_ids = [s.scene_id for s in candidates]
        scene_ids = [sid for sid in scene_ids if sid in valid_ids]

        # Enforce diversity: max 2 scenes from the same source video
        scene_ids = enforce_diversity(scene_ids)

        # Ensure we have enough scenes
        if len(scene_ids) < MIN_SCENES:
            # Add more from candidates
            for s in candidates:
                if s.scene_id not in scene_ids:
                    scene_ids.append(s.scene_id)
                    scene_ids = enforce_diversity(scene_ids)
                if len(scene_ids) >= MIN_SCENES:
                    break

        return scene_ids[:MAX_SCENES], prompt

    except Exception as e:
        logger.error(f"Error calling Claude: {e}")
        # Fallback: return top candidates by mood overlap
        fallback = [s.scene_id for s in candidates[:MAX_SCENES]]
        return enforce_diversity(fallback), prompt

def select_scenes_baseline(scenes: List[SceneData],
                           num_candidates: int = 30) -> Tuple[List[str], str]:
    """
    Select scenes using baseline method (random but coherent).
    
    Args:
        scenes: Pool of available scenes
        num_candidates: Number of candidates to show Claude
    
    Returns:
        Tuple of (ordered scene IDs, prompt used)
    """
    # Random selection of candidates
    candidates = random.sample(scenes, min(num_candidates, len(scenes)))

    # Build scene list for prompt
    scene_list = "\n".join([
        f"{s.scene_id}: {', '.join(s.style_words)}"
        for s in candidates
    ])

    # Format prompt
    prompt = BASELINE_PROMPT.format(scene_list=scene_list)

    # Call Claude
    try:
        message = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}]
        )
        response = message.content[0].text.strip()

        # Parse scene IDs
        scene_ids = [s.strip() for s in response.split(',')]

        # Validate scene IDs
        valid_ids = [s.scene_id for s in candidates]
        scene_ids = [sid for sid in scene_ids if sid in valid_ids]

        # Enforce diversity: max 2 scenes from the same source video
        scene_ids = enforce_diversity(scene_ids)

        # Ensure we have enough scenes
        if len(scene_ids) < MIN_SCENES:
            for s in candidates:
                if s.scene_id not in scene_ids:
                    scene_ids.append(s.scene_id)
                    scene_ids = enforce_diversity(scene_ids)
                if len(scene_ids) >= MIN_SCENES:
                    break

        return scene_ids[:MAX_SCENES], prompt

    except Exception as e:
        logger.error(f"Error calling Claude: {e}")
        # Fallback: random selection
        fallback = [s.scene_id for s in candidates[:MAX_SCENES]]
        return enforce_diversity(fallback), prompt

logger.info("Scene selection functions defined.")

## 5. Video Rendering

In [ ]:
def get_scene_time_range(scene_id: str) -> Tuple[float, float]:
    """
    Get the time range for a scene from scene metadata.
    
    Args:
        scene_id: Scene identifier (format: {video_id}_s{index})
    
    Returns:
        Tuple of (start_time, end_time) in seconds
    """
    # Parse video_id from scene_id
    parts = scene_id.rsplit('_s', 1)
    if len(parts) != 2:
        return 0, 5  # Default 5 second clip

    video_id = parts[0]

    # Load scene metadata
    metadata_path = SCENES_DIR / video_id / "scene_metadata.json"
    if not metadata_path.exists():
        # Try to infer from keyframe (default 3 second clip)
        return 0, 3

    with open(metadata_path, 'r') as f:
        scenes = json.load(f)

    for scene in scenes:
        if scene['scene_id'] == scene_id:
            return scene['start_time'], scene['end_time']

    return 0, 3  # Default

def get_video_path(scene_id: str) -> Optional[Path]:
    """
    Get the source video path for a scene.
    """
    # Parse video_id from scene_id
    parts = scene_id.rsplit('_s', 1)
    if len(parts) != 2:
        return None

    video_id = parts[0]

    # Load manifest
    manifest_path = BASE_DIR / "video_manifest.csv"
    manifest_df = pd.read_csv(manifest_path)
    manifest_df['local_path'] = manifest_df['local_path'].str.replace('\\\\', '/', regex=False).str.replace('\\', '/', regex=False)

    # Find video
    video_row = manifest_df[manifest_df['video_id'] == video_id]
    if video_row.empty:
        return None

    video_path = BASE_DIR / video_row.iloc[0]['local_path']
    return video_path if video_path.exists() else None

print("Video helper functions defined.")

In [ ]:
def render_video(scene_ids: List[str],
                 output_path: Path,
                 target_duration: float = TARGET_DURATION) -> Tuple[Optional[float], List[str]]:
    """
    Render a video by concatenating scenes with duration fitting.
    
    Args:
        scene_ids: Ordered list of scene IDs
        output_path: Path to save the output video
        target_duration: Target video duration in seconds
    
    Returns:
        Tuple of (actual duration or None if failed, list of skipped scene IDs)
    """
    clips = []
    skipped_scenes = []

    # Duration fitting: track remaining budget
    remaining = target_duration
    scenes_left = len(scene_ids)

    for scene_id in tqdm(scene_ids, desc="Loading clips"):
        video_path = get_video_path(scene_id)
        if video_path is None:
            logger.warning(f"Skipping {scene_id}: video file not found")
            skipped_scenes.append(scene_id)
            scenes_left -= 1
            continue

        start_time, end_time = get_scene_time_range(scene_id)

        # Calculate clip duration to fit remaining budget
        clip_dur = min(end_time - start_time, remaining / max(1, scenes_left))
        end_time = start_time + clip_dur

        try:
            clip = VideoFileClip(str(video_path)).subclip(start_time, end_time)

            # Add fade transitions
            if len(clips) > 0:
                clip = fadein(clip, TRANSITION_DURATION)
            clip = fadeout(clip, TRANSITION_DURATION)

            clips.append(clip)
            remaining -= clip.duration
            scenes_left -= 1
        except Exception as e:
            logger.warning(f"Skipping {scene_id}: {e}")
            skipped_scenes.append(scene_id)
            scenes_left -= 1
            continue

    if not clips:
        logger.error("No clips to render.")
        return None, skipped_scenes

    # Concatenate
    logger.info(f"Concatenating {len(clips)} clips...")
    final = concatenate_videoclips(clips, method="compose")

    # Duration deviation check
    if abs(final.duration - target_duration) > 3:
        logger.warning(f"Duration {final.duration:.1f}s deviates from target {target_duration}s")

    # Export
    output_path.parent.mkdir(parents=True, exist_ok=True)
    logger.info(f"Exporting to {output_path}...")

    final.write_videofile(
        str(output_path),
        fps=OUTPUT_FPS,
        codec='libx264',
        audio_codec='aac',
        logger='bar'
    )

    duration = final.duration

    # Cleanup
    for clip in clips:
        clip.close()
    final.close()

    logger.info(f"Video saved: {output_path} ({duration:.1f}s)")
    if skipped_scenes:
        logger.info(f"Skipped {len(skipped_scenes)} scenes: {skipped_scenes}")
    return duration, skipped_scenes

logger.info("Video rendering function defined.")

## 6. Generate Videos for Each Cluster

In [ ]:
def generate_gacs_video(cluster_id: int,
                        cluster_info: Dict,
                        scenes: List[SceneData]) -> Optional[GeneratedVideoMetadata]:
    """
    Generate a GACS video for a mood cluster.
    
    Args:
        cluster_id: Cluster index
        cluster_info: Info dict for this cluster
        scenes: All available scenes
    
    Returns:
        GeneratedVideoMetadata or None if failed
    """
    logger.info(f"Generating GACS video for Cluster {cluster_id}")
    logger.info(f"Target mood: {cluster_info['top_mood_words']}")

    # Select scenes using GACS method
    scene_ids, prompt_used = select_scenes_gacs(
        cluster_info['top_mood_words'],
        scenes
    )

    logger.info(f"Selected {len(scene_ids)} scenes: {scene_ids}")

    # Generate video ID
    video_id = f"gacs_cluster{cluster_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    output_path = GACS_OUTPUT_DIR / f"{video_id}.mp4"

    # Render video
    duration, skipped_scenes = render_video(scene_ids, output_path)

    if duration is None:
        return None

    # Create metadata
    metadata = GeneratedVideoMetadata(
        video_id=video_id,
        group="gacs",
        mood_vector=cluster_info['centroid'].tolist(),
        mood_words=cluster_info['top_mood_words'],
        prompt_used=prompt_used,
        source_scenes=scene_ids,
        created_at=datetime.now().isoformat(),
        duration=duration
    )

    # Enrich with render stats
    metadata_dict = asdict(metadata)
    metadata_dict['render_stats'] = {
        'scenes_requested': len(scene_ids),
        'scenes_used': len(scene_ids) - len(skipped_scenes),
        'scenes_skipped': len(skipped_scenes),
        'actual_duration': duration,
        'target_duration': TARGET_DURATION,
        'duration_error': abs(duration - TARGET_DURATION)
    }

    # Save metadata
    metadata_path = GACS_OUTPUT_DIR / f"{video_id}" / "metadata.json"
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    with open(metadata_path, 'w') as f:
        json.dump(metadata_dict, f, indent=2)

    # Also copy video to metadata folder
    import shutil
    shutil.copy(output_path, metadata_path.parent / "video.mp4")

    logger.info(f"Metadata saved: {metadata_path}")

    return metadata

def generate_baseline_video(cluster_id: int,
                            scenes: List[SceneData]) -> Optional[GeneratedVideoMetadata]:
    """
    Generate a baseline (random) video.
    
    Args:
        cluster_id: Cluster index (for naming only)
        scenes: All available scenes
    
    Returns:
        GeneratedVideoMetadata or None if failed
    """
    logger.info(f"Generating Baseline video {cluster_id}")

    # Select scenes using baseline method
    scene_ids, prompt_used = select_scenes_baseline(scenes)

    logger.info(f"Selected {len(scene_ids)} scenes: {scene_ids}")

    # Generate video ID
    video_id = f"baseline_{cluster_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    output_path = BASELINE_OUTPUT_DIR / f"{video_id}.mp4"

    # Render video
    duration, skipped_scenes = render_video(scene_ids, output_path)

    if duration is None:
        return None

    # Create metadata
    metadata = GeneratedVideoMetadata(
        video_id=video_id,
        group="baseline",
        mood_vector=[],  # No target mood for baseline
        mood_words=[],
        prompt_used=prompt_used,
        source_scenes=scene_ids,
        created_at=datetime.now().isoformat(),
        duration=duration
    )

    # Enrich with render stats
    metadata_dict = asdict(metadata)
    metadata_dict['render_stats'] = {
        'scenes_requested': len(scene_ids),
        'scenes_used': len(scene_ids) - len(skipped_scenes),
        'scenes_skipped': len(skipped_scenes),
        'actual_duration': duration,
        'target_duration': TARGET_DURATION,
        'duration_error': abs(duration - TARGET_DURATION)
    }

    # Save metadata
    metadata_path = BASELINE_OUTPUT_DIR / f"{video_id}" / "metadata.json"
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    with open(metadata_path, 'w') as f:
        json.dump(metadata_dict, f, indent=2)

    # Also copy video
    import shutil
    shutil.copy(output_path, metadata_path.parent / "video.mp4")

    logger.info(f"Metadata saved: {metadata_path}")

    return metadata

logger.info("Video generation functions defined.")

In [ ]:
# Generate videos for each cluster
all_gacs_videos = []
all_baseline_videos = []

if scenes_data and cluster_info:
    clusters_to_process = range(len(cluster_info))
    if QUICK_TEST_MODE:
        clusters_to_process = range(min(1, len(cluster_info)))
        logger.info("QUICK_TEST_MODE: Processing only 1 cluster")

    for cluster_id in clusters_to_process:
        # Generate GACS video
        gacs_meta = generate_gacs_video(cluster_id, cluster_info[cluster_id], scenes_data)
        if gacs_meta:
            all_gacs_videos.append(gacs_meta)

        # Generate corresponding baseline video
        baseline_meta = generate_baseline_video(cluster_id, scenes_data)
        if baseline_meta:
            all_baseline_videos.append(baseline_meta)

    logger.info(f"Generation Complete! GACS videos: {len(all_gacs_videos)}, Baseline videos: {len(all_baseline_videos)}")
else:
    logger.warning("No data available. Run gacs_dataset_builder.ipynb first.")

## 7. List Generated Videos

In [ ]:
def list_generated_videos():
    """
    List all generated videos with metadata.
    """
    videos = []

    # Check GACS videos
    for video_dir in GACS_OUTPUT_DIR.iterdir():
        if video_dir.is_dir():
            metadata_path = video_dir / "metadata.json"
            if metadata_path.exists():
                with open(metadata_path, 'r') as f:
                    meta = json.load(f)
                videos.append({
                    'video_id': meta['video_id'],
                    'group': meta['group'],
                    'mood_words': ', '.join(meta.get('mood_words', [])[:3]),
                    'duration': f"{meta['duration']:.1f}s",
                    'num_scenes': len(meta['source_scenes']),
                    'path': str(video_dir / 'video.mp4')
                })

    # Check baseline videos
    for video_dir in BASELINE_OUTPUT_DIR.iterdir():
        if video_dir.is_dir():
            metadata_path = video_dir / "metadata.json"
            if metadata_path.exists():
                with open(metadata_path, 'r') as f:
                    meta = json.load(f)
                videos.append({
                    'video_id': meta['video_id'],
                    'group': meta['group'],
                    'mood_words': '(random)',
                    'duration': f"{meta['duration']:.1f}s",
                    'num_scenes': len(meta['source_scenes']),
                    'path': str(video_dir / 'video.mp4')
                })

    if videos:
        df = pd.DataFrame(videos)
        display(df)
    else:
        print("No generated videos found.")

    return videos

generated = list_generated_videos()

## 8. Summary

This notebook has:
1. Loaded GACS dataset from `data/embeddings/gacs_dataset.csv`
2. Clustered mood vectors into K clusters (default K=5)
3. For each cluster, generated:
   - **GACS video**: Mood-based scene selection using the GACS prompt
   - **Baseline video**: Random scene selection for comparison
4. Rendered 30-second videos with fade transitions
5. Saved videos and metadata

**Output Structure:**
```
data/generated/
├── gacs/
│   ├── {video_id}.mp4
│   └── {video_id}/
│       ├── video.mp4
│       └── metadata.json
└── baseline/
    ├── {video_id}.mp4
    └── {video_id}/
        ├── video.mp4
        └── metadata.json
```

**Metadata Schema:**
```json
{
  "video_id": "string",
  "group": "gacs" | "baseline",
  "mood_vector": [float],
  "mood_words": [string],
  "prompt_used": "string",
  "source_scenes": [string],
  "created_at": "ISO timestamp",
  "duration": float
}
```

**Next Step:** Run `youtube_experiment_runner.ipynb` to upload videos and collect metrics.